# 🛡️ IoT IDS — Two-Stage Attack Detection Pipeline
**Team:** Jinhong Lin · Elton Chang · Shiwei Jiang

## Architecture
```
Network Flow
     │
     ▼
┌─────────────────────────────────┐
│  STAGE 1 — Binary Classifier    │  Is this traffic an attack?
│  Models: LogReg · RF · HGB      │
│          MLP · 1D CNN           │  0 = Attack  /  1 = Normal
└─────────────┬───────────────────┘
              │ if ATTACK (0)
              ▼
┌─────────────────────────────────┐
│  STAGE 2 — Attack Type ID       │  Which attack is it?
│  Model: HGB + 1D CNN            │
│                                 │  udp · tcp · syn · ack
└─────────────────────────────────┘  combo · junk · scan · udpplain
```

## Output files saved to `/kaggle/working/`
| File | Contents |
|------|----------|
| `stage1_binary_results.csv` | F1 / AUC / latency for all 5 Stage-1 models |
| `stage1_per_class_report.csv` | Precision / Recall / F1 per class per model |
| `stage1_predictions.csv` | Full test predictions from all 5 Stage-1 models |
| `stage2_family_report.csv` | Normal / Mirai / Gafgyt classification report |
| `stage2_subtype_report.csv` | 9-subtype classification report |
| `stage2_predictions.csv` | Stage-2 predictions on attack-flagged samples |
| `pipeline_combined_predictions.csv` | Final end-to-end pipeline output per sample |
| `feature_importance_rf.csv` | RF feature importances (ranked) |
| `per_device_results.csv` | Per-device F1 scores |
| `cnn_training_history.csv` | CNN loss / AUC per epoch (Stage 1 + Stage 2) |


## Section 0 — Setup

In [2]:
# All packages are pre-installed on Kaggle
import os, time, warnings, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers, callbacks as K_callbacks
    TF_AVAILABLE = True
except ImportError:
    print("⚠️  TensorFlow not available - CNN models will be skipped")
    TF_AVAILABLE = False

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, precision_score, recall_score,
    roc_auc_score, ConfusionMatrixDisplay
)
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
if TF_AVAILABLE:
    tf.get_logger().setLevel("ERROR")
    tf.random.set_seed(42)
np.random.seed(42)
plt.rcParams.update({"figure.dpi": 110,
                     "axes.spines.top": False,
                     "axes.spines.right": False})

# ── Paths ─────────────────────────────────────────────────────────
DATA_DIR = Path(r"C:\Users\henda\Downloads\iot_data")
OUT      = Path(r"C:\Users\henda\Downloads\IoT_Results")
OUT.mkdir(exist_ok=True)

CSV1 = DATA_DIR / "BotNeTIoT-L01_label_NoDuplicates.csv"
CSV2 = DATA_DIR / "BoTNeTIoT-L01-v2.csv"

# ── Config ────────────────────────────────────────────────────────
SAMPLE_N     = 400_000   # rows from Dataset 1 (binary)
SAMPLE_N2    = 500_000   # rows from Dataset 2 (multiclass)
RANDOM_STATE = 42
TEST_SIZE    = 0.15
VAL_FRAC     = 0.15 / (1 - 0.15)

if TF_AVAILABLE:
    print(f"TensorFlow : {tf.__version__}")
print(f"Output dir : {OUT}")
print("Input files:")
for f in sorted(DATA_DIR.glob("*.csv")):
    print(f"  {f.name}  ({f.stat().st_size/1e6:.0f} MB)")

⚠️  TensorFlow not available - CNN models will be skipped
Output dir : C:\Users\henda\Downloads\IoT_Results
Input files:
  BoTNeTIoT-L01-v2.csv  (1678 MB)
  BotNeTIoT-L01_label_NoDuplicates.csv  (636 MB)


## Section 1 — Data Loading

In [3]:
def stratified_sample_csv(path, n_samples, label_col="label",
                           chunksize=150_000, random_state=42):
    path = Path(path)
    print(f"  📂 {path.name}  ({path.stat().st_size/1e6:.0f} MB)")
    if n_samples is None:
        df = pd.read_csv(path)
        print(f"  ✅ {len(df):,} rows (full)")
        return df

    print("  Scanning labels …")
    all_labels = pd.concat(
        [chunk[label_col] for chunk in
         pd.read_csv(path, usecols=[label_col], chunksize=chunksize)],
        ignore_index=True).values

    classes, counts = np.unique(all_labels, return_counts=True)
    fracs = counts / counts.sum()
    rng   = np.random.RandomState(random_state)
    chosen = set()
    for c, f in zip(classes, fracs):
        idx_c = np.where(all_labels == c)[0]
        n_c   = min(int(n_samples * f), len(idx_c))
        chosen.update(rng.choice(idx_c, size=n_c, replace=False).tolist())

    print(f"  Reading {len(chosen):,} selected rows …")
    chunks_out, row_ptr = [], 0
    for chunk in pd.read_csv(path, chunksize=chunksize):
        mask = np.array([(row_ptr + i) in chosen for i in range(len(chunk))])
        if mask.any():
            chunks_out.append(chunk[mask])
        row_ptr += len(chunk)

    df = pd.concat(chunks_out, ignore_index=True)
    dist = {int(c): int(n) for c, n in
            zip(*np.unique(df[label_col], return_counts=True))}
    print(f"  ✅ {len(df):,} rows  |  classes: {dist}")
    return df


print("Loading Dataset 1 (binary labels) …")
df1 = stratified_sample_csv(CSV1 if CSV1.exists() else CSV2, SAMPLE_N)

print("\nLoading Dataset 2 (with attack metadata) …")
df2 = stratified_sample_csv(CSV2, SAMPLE_N2)

# Drop index column
for _df in [df1, df2]:
    if "Unnamed: 0" in _df.columns:
        _df.drop(columns=["Unnamed: 0"], inplace=True)

print(f"\n✅ df1: {df1.shape}  |  df2: {df2.shape}")


Loading Dataset 1 (binary labels) …
  📂 BotNeTIoT-L01_label_NoDuplicates.csv  (636 MB)
  Scanning labels …
  Reading 399,999 selected rows …
  ✅ 399,999 rows  |  classes: {0: 315354, 1: 84645}

Loading Dataset 2 (with attack metadata) …
  📂 BoTNeTIoT-L01-v2.csv  (1678 MB)
  Scanning labels …
  Reading 499,999 selected rows …
  ✅ 499,999 rows  |  classes: {0: 460642, 1: 39357}

✅ df1: (399999, 24)  |  df2: (499999, 27)


## Section 2 — Preprocessing

In [4]:
FEATURE_COLS = [
    "MI_dir_L0.1_weight","MI_dir_L0.1_mean","MI_dir_L0.1_variance",
    "H_L0.1_weight","H_L0.1_mean","H_L0.1_variance",
    "HH_L0.1_weight","HH_L0.1_mean","HH_L0.1_std",
    "HH_L0.1_magnitude","HH_L0.1_radius","HH_L0.1_covariance","HH_L0.1_pcc",
    "HH_jit_L0.1_weight","HH_jit_L0.1_mean","HH_jit_L0.1_variance",
    "HpHp_L0.1_weight","HpHp_L0.1_mean","HpHp_L0.1_std",
    "HpHp_L0.1_magnitude","HpHp_L0.1_radius","HpHp_L0.1_covariance","HpHp_L0.1_pcc",
]

# Drop 4 perfectly redundant features (r = 1.00)
DROP_REDUNDANT = ["H_L0.1_weight","H_L0.1_mean","H_L0.1_variance","HH_jit_L0.1_weight"]
USE_FEATS = [f for f in FEATURE_COLS if f not in DROP_REDUNDANT]
print(f"Features: {len(FEATURE_COLS)} → {len(USE_FEATS)} (dropped {len(DROP_REDUNDANT)} redundant)")

# log1p transform helper (fit on train only)
SKEW_FEATS = [f for f in USE_FEATS if abs(df1[f].skew()) > 1]
print(f"Applying log1p to {len(SKEW_FEATS)} skewed features")

def log1p_tf(X_df, cols):
    X = X_df.copy()
    for c in cols:
        if c in X.columns:
            X[c] = np.log1p(np.abs(X[c]))
    return X

# ── Stage 1 splits (from Dataset 1 / binary) ─────────────────────
X_all = df1[USE_FEATS].copy()
y_all = df1["label"].copy()

X_tv, X_test, y_tv, y_test = train_test_split(
    X_all, y_all, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_all)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=VAL_FRAC, random_state=RANDOM_STATE, stratify=y_tv)

scaler1 = StandardScaler()
X_train_s = scaler1.fit_transform(log1p_tf(X_train, SKEW_FEATS))
X_val_s   = scaler1.transform(log1p_tf(X_val,   SKEW_FEATS))
X_test_s  = scaler1.transform(log1p_tf(X_test,  SKEW_FEATS))

print(f"\nStage 1 splits → Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")

# ── Stage 2 splits (from Dataset 2 / multiclass) ─────────────────
use2 = [f for f in USE_FEATS if f in df2.columns]
X2_all  = log1p_tf(df2[use2], SKEW_FEATS)
y2_sub  = df2["Attack_subType"] if "Attack_subType" in df2.columns else None
y2_fam  = df2["Attack"]         if "Attack" in df2.columns else None

if y2_sub is not None:
    X2tv, X2te, y2tv, y2te = train_test_split(
        X2_all, y2_sub, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y2_sub)
    X2tr, X2val, y2tr, y2val = train_test_split(
        X2tv, y2tv, test_size=VAL_FRAC, random_state=RANDOM_STATE, stratify=y2tv)

    scaler2 = StandardScaler()
    X2tr_s  = scaler2.fit_transform(X2tr)
    X2val_s = scaler2.transform(X2val)
    X2te_s  = scaler2.transform(X2te)

    le = LabelEncoder()
    y2tr_enc  = le.fit_transform(y2tr)
    y2val_enc = le.transform(y2val)
    y2te_enc  = le.transform(y2te)
    N_CLASSES2 = len(le.classes_)
    print(f"Stage 2 splits → Train: {len(X2tr):,} | Test: {len(X2te):,}")
    print(f"Stage 2 classes ({N_CLASSES2}): {list(le.classes_)}")

print("\n✅ Preprocessing complete")


Features: 23 → 19 (dropped 4 redundant)
Applying log1p to 10 skewed features

Stage 1 splits → Train: 279,999 | Val: 60,000 | Test: 60,000
Stage 2 splits → Train: 349,999 | Test: 75,000
Stage 2 classes (9): ['Normal', 'ack', 'combo', 'junk', 'scan', 'syn', 'tcp', 'udp', 'udpplain']

✅ Preprocessing complete


## Section 3 — 1D CNN Architecture

The CNN treats each sample's **19 flow statistics as a 1D sequence** of length 19.
Convolutional filters slide across neighboring features to learn co-occurring patterns
(e.g. high packet weight + high jitter variance = flood attack signature).

```
Input (19 features) → reshape → (19, 1)
  ┌─ Conv1D(64,  k=3, padding=same) → BatchNorm → ReLU ─┐
  ├─ Conv1D(128, k=3, padding=same) → BatchNorm → ReLU ─┤  Feature extraction
  └─ Conv1D(64,  k=3, padding=same) → BatchNorm → ReLU ─┘
  GlobalAveragePooling1D
  Dense(64, ReLU) → Dropout(0.3)
  Dense(n_classes, Softmax / Sigmoid) → prediction
```


In [5]:
def build_cnn_binary(n_features):
    """Stage 1 CNN — binary: attack(0) vs normal(1)."""
    inp = keras.Input(shape=(n_features, 1), name="flow_input")
    x = layers.Conv1D(64,  3, padding="same", name="conv1")(inp)
    x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    x = layers.Conv1D(128, 3, padding="same", name="conv2")(x)
    x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    x = layers.Conv1D(64,  3, padding="same", name="conv3")(x)
    x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(1, activation="sigmoid", name="output")(x)
    model = keras.Model(inp, out, name="CNN_Stage1_Binary")
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy",
                 keras.metrics.AUC(name="auc"),
                 keras.metrics.Precision(name="precision"),
                 keras.metrics.Recall(name="recall")])
    return model


def build_cnn_multiclass(n_features, n_classes):
    """Stage 2 CNN — multiclass: attack subtype identification."""
    inp = keras.Input(shape=(n_features, 1), name="flow_input")
    x = layers.Conv1D(64,  3, padding="same", name="conv1")(inp)
    x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    x = layers.Conv1D(128, 3, padding="same", name="conv2")(x)
    x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    x = layers.Conv1D(128, 3, padding="same", name="conv3")(x)
    x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    x = layers.Conv1D(64,  3, padding="same", name="conv4")(x)
    x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64,  activation="relu")(x)
    x = layers.Dropout(0.2)(x)
    out = layers.Dense(n_classes, activation="softmax", name="output")(x)
    model = keras.Model(inp, out, name="CNN_Stage2_Multiclass")
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy",
                 keras.metrics.SparseTopKCategoricalAccuracy(k=2, name="top2_acc")])
    return model


cnn_s1 = build_cnn_binary(X_train_s.shape[1])
cnn_s1.summary()


NameError: name 'keras' is not defined

## Section 4 — Stage 1: Binary Classification (Attack vs Normal)
Train all 5 models. The best model's predictions feed into Stage 2.


In [6]:
cw  = compute_class_weight("balanced", classes=np.array([0,1]), y=y_train)
cwd = {0: float(cw[0]), 1: float(cw[1])}
print(f"Class weights → Attack(0): {cw[0]:.3f}  Normal(1): {cw[1]:.3f}")

sklearn_models = {
    "LogReg": LogisticRegression(max_iter=500, random_state=RANDOM_STATE,
               class_weight="balanced", solver="lbfgs", C=1.0),
    "RF":     RandomForestClassifier(n_estimators=100, max_depth=20,
               random_state=RANDOM_STATE, class_weight="balanced", n_jobs=-1),
    "HGB":    HistGradientBoostingClassifier(max_iter=150, max_depth=8,
               learning_rate=0.05, random_state=RANDOM_STATE, class_weight="balanced"),
    "MLP":    MLPClassifier(hidden_layer_sizes=(256,128,64), activation="relu",
               solver="adam", alpha=1e-4, batch_size=1024, learning_rate_init=1e-3,
               max_iter=30, early_stopping=True, validation_fraction=0.1,
               n_iter_no_change=5, random_state=RANDOM_STATE),
}

# ── Train sklearn models ──────────────────────────────────────────
s1_val = {}
for name, model in sklearn_models.items():
    print(f"\n── {name} {'─'*(35-len(name))}")
    t0 = time.time()
    model.fit(X_train_s, y_train)
    train_time = time.time() - t0

    y_pred  = model.predict(X_val_s)
    y_proba = (model.predict_proba(X_val_s)[:,1]
               if hasattr(model, "predict_proba")
               else model.decision_function(X_val_s))

    t1 = time.time(); _ = model.predict(X_val_s)
    lat = (time.time()-t1)/len(X_val_s)*1e6

    s1_val[name] = dict(
        model=model,
        f1_attack  = f1_score(y_val, y_pred, pos_label=0),
        f1_normal  = f1_score(y_val, y_pred, pos_label=1),
        f1_macro   = f1_score(y_val, y_pred, average="macro"),
        auc_roc    = roc_auc_score(y_val, y_proba),
        latency_us = lat, train_time = train_time,
    )
    print(f"  F1-Macro: {s1_val[name]['f1_macro']:.4f}  "
          f"AUC: {s1_val[name]['auc_roc']:.4f}  Latency: {lat:.2f}μs")

print("\n✅ Sklearn Stage-1 models trained")


Class weights → Attack(0): 0.634  Normal(1): 2.363

── LogReg ─────────────────────────────
  F1-Macro: 0.9990  AUC: 0.9999  Latency: 0.02μs

── RF ─────────────────────────────────
  F1-Macro: 0.9999  AUC: 1.0000  Latency: 0.67μs

── HGB ────────────────────────────────
  F1-Macro: 0.9999  AUC: 1.0000  Latency: 1.03μs

── MLP ────────────────────────────────
  F1-Macro: 0.9996  AUC: 0.9999  Latency: 1.38μs

✅ Sklearn Stage-1 models trained


In [7]:
# ── Train Stage 1 CNN ─────────────────────────────────────────────
X_tr_cnn  = X_train_s.reshape(-1, X_train_s.shape[1], 1)
X_val_cnn = X_val_s.reshape(-1,   X_val_s.shape[1],   1)
X_te_cnn  = X_test_s.reshape(-1,  X_test_s.shape[1],  1)

cnn_s1_callbacks = [
    K_callbacks.EarlyStopping(patience=4, restore_best_weights=True,
                              monitor="val_auc", mode="max"),
    K_callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                  patience=2, min_lr=1e-5),
]

print("Training Stage 1 CNN …")
t0 = time.time()
hist_s1 = cnn_s1.fit(
    X_tr_cnn, y_train.values,
    validation_data=(X_val_cnn, y_val.values),
    epochs=30, batch_size=1024,
    class_weight=cwd, callbacks=cnn_s1_callbacks, verbose=1,
)
cnn_s1_time = time.time() - t0
print(f"\nCNN Stage-1 training: {cnn_s1_time:.1f}s  ({len(hist_s1.epoch)} epochs)")

cnn_val_proba = cnn_s1.predict(X_val_cnn, verbose=0).ravel()
cnn_val_pred  = (cnn_val_proba >= 0.5).astype(int)

t1 = time.time()
_ = cnn_s1.predict(X_val_cnn[:500], verbose=0)
cnn_lat = (time.time()-t1)/500*1e6

s1_val["CNN"] = dict(
    model      = cnn_s1,
    f1_attack  = f1_score(y_val, cnn_val_pred, pos_label=0),
    f1_normal  = f1_score(y_val, cnn_val_pred, pos_label=1),
    f1_macro   = f1_score(y_val, cnn_val_pred, average="macro"),
    auc_roc    = roc_auc_score(y_val, cnn_val_proba),
    latency_us = cnn_lat, train_time = cnn_s1_time,
)
print(f"CNN Stage-1 → F1-Macro: {s1_val['CNN']['f1_macro']:.4f}  "
      f"AUC: {s1_val['CNN']['auc_roc']:.4f}  Latency: {cnn_lat:.2f}μs")


NameError: name 'K_callbacks' is not defined

In [ ]:
# ── Stage 1 Training Curves ───────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric, title in zip(axes,
        ["loss",     "auc",     "recall"],
        ["Loss",     "AUC",     "Recall"]):
    axes[list(axes).index(ax)].plot(hist_s1.history[metric],        label="train", color="#E74C3C")
    axes[list(axes).index(ax)].plot(hist_s1.history[f"val_{metric}"],label="val",   color="#3498DB")
    axes[list(axes).index(ax)].set_title(f"CNN Stage 1 — {title}", fontweight="bold")
    axes[list(axes).index(ax)].legend()

plt.suptitle("Stage 1 CNN Training Curves", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT/"cnn_stage1_training_curves.png", dpi=120, bbox_inches="tight")
plt.show(); print("Saved cnn_stage1_training_curves.png")


In [8]:
# ── Stage 1 Test Evaluation ───────────────────────────────────────
s1_test = {}
latency_map = {}
N_BENCH = min(1000, len(X_test_s))
X_bench = X_test_s[:N_BENCH]

# Sklearn models
for name, res in s1_val.items():
    if name == "CNN":
        continue
    model = res["model"]
    y_pred  = model.predict(X_test_s)
    y_proba = (model.predict_proba(X_test_s)[:,1]
               if hasattr(model,"predict_proba")
               else model.decision_function(X_test_s))

    model.predict(X_bench[:10])
    t0 = time.perf_counter()
    for _ in range(5): model.predict(X_bench)
    latency_map[name] = (time.perf_counter()-t0)/(5*N_BENCH)*1e6

    s1_test[name] = dict(
        f1_attack  = f1_score(y_test, y_pred, pos_label=0),
        f1_normal  = f1_score(y_test, y_pred, pos_label=1),
        f1_macro   = f1_score(y_test, y_pred, average="macro"),
        auc_roc    = roc_auc_score(y_test, y_proba),
        latency_us = latency_map[name],
        y_pred     = y_pred, y_proba = y_proba,
    )

print("\n── STAGE 1 TEST RESULTS ──────────────────────────────────────")
for name in s1_test:
    print(f"\n{name}:")
    print(classification_report(y_test, s1_test[name]["y_pred"],
          target_names=["Attack(0)","Normal(1)"]))


── STAGE 1 TEST RESULTS ──────────────────────────────────────

LogReg:
              precision    recall  f1-score   support

   Attack(0)       1.00      1.00      1.00     47303
   Normal(1)       1.00      1.00      1.00     12697

    accuracy                           1.00     60000
   macro avg       1.00      1.00      1.00     60000
weighted avg       1.00      1.00      1.00     60000


RF:
              precision    recall  f1-score   support

   Attack(0)       1.00      1.00      1.00     47303
   Normal(1)       1.00      1.00      1.00     12697

    accuracy                           1.00     60000
   macro avg       1.00      1.00      1.00     60000
weighted avg       1.00      1.00      1.00     60000


HGB:
              precision    recall  f1-score   support

   Attack(0)       1.00      1.00      1.00     47303
   Normal(1)       1.00      1.00      1.00     12697

    accuracy                           1.00     60000
   macro avg       1.00      1.00      1.00 

In [9]:
# ════════════════════════════════════════════════════════════
# SAVE: stage1_binary_results.csv
# ════════════════════════════════════════════════════════════
rows = []
for name in s1_test:
    rows.append({
        "model":         name,
        "stage":         "Stage1_Binary",
        "f1_attack_0":   round(s1_test[name]["f1_attack"],  4),
        "f1_normal_1":   round(s1_test[name]["f1_normal"],  4),
        "f1_macro":      round(s1_test[name]["f1_macro"],   4),
        "auc_roc":       round(s1_test[name]["auc_roc"],    4),
        "latency_us":    round(latency_map[name],           3),
        "train_time_s":  round(s1_val[name]["train_time"],  1),
    })
df_s1 = pd.DataFrame(rows)
df_s1.to_csv(OUT/"stage1_binary_results.csv", index=False)
print("✅ Saved stage1_binary_results.csv")
print(df_s1.to_string(index=False))


✅ Saved stage1_binary_results.csv
 model         stage  f1_attack_0  f1_normal_1  f1_macro  auc_roc  latency_us  train_time_s
LogReg Stage1_Binary       0.9998       0.9991    0.9994      1.0       0.073           0.3
    RF Stage1_Binary       1.0000       0.9999    0.9999      1.0      25.597           2.4
   HGB Stage1_Binary       1.0000       0.9999    0.9999      1.0       1.822           2.6
   MLP Stage1_Binary       0.9999       0.9997    0.9998      1.0       1.113          15.4


In [10]:
# ════════════════════════════════════════════════════════════
# SAVE: stage1_per_class_report.csv
# ════════════════════════════════════════════════════════════
report_rows = []
for name in s1_test:
    y_pred = s1_test[name]["y_pred"]
    for lv, ln in [(0,"Attack(0)"), (1,"Normal(1)")]:
        report_rows.append({
            "model":     name,
            "class":     ln,
            "precision": round(precision_score(y_test, y_pred, pos_label=lv, average="binary"), 4),
            "recall":    round(recall_score(y_test, y_pred, pos_label=lv, average="binary"),    4),
            "f1_score":  round(f1_score(y_test, y_pred, pos_label=lv, average="binary"),        4),
            "support":   int((y_test == lv).sum()),
        })
pd.DataFrame(report_rows).to_csv(OUT/"stage1_per_class_report.csv", index=False)
print("✅ Saved stage1_per_class_report.csv")


✅ Saved stage1_per_class_report.csv


In [11]:
# ════════════════════════════════════════════════════════════
# SAVE: stage1_predictions.csv
# ════════════════════════════════════════════════════════════
pred_df = X_test.copy().reset_index(drop=True)
pred_df["true_label"] = y_test.values
for name in s1_test:
    pred_df[f"pred_{name}"]  = s1_test[name]["y_pred"]
    pred_df[f"proba_{name}"] = s1_test[name]["y_proba"].round(4)
pred_df.to_csv(OUT/"stage1_predictions.csv", index=False)
print(f"✅ Saved stage1_predictions.csv  ({len(pred_df):,} rows × {pred_df.shape[1]} cols)")

# ── Comparison plot ────────────────────────────────────────────────────
names = list(s1_test.keys())
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
x = np.arange(len(names)); w = 0.35
axes[0].bar(x-w/2, [s1_test[n]["f1_attack"] for n in names], w,
            label="Attack(0)", color="#E74C3C", alpha=0.85)
axes[0].bar(x+w/2, [s1_test[n]["f1_normal"] for n in names], w,
            label="Normal(1)", color="#2ECC71", alpha=0.85)
axes[0].set_xticks(x); axes[0].set_xticklabels(names)
axes[0].set_ylim(0.96, 1.005); axes[0].set_ylabel("F1-Score")
axes[0].set_title("Stage 1 — Per-Class F1 (test)", fontweight="bold")
axes[0].legend(); axes[0].yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))

clrs = ["#3498DB","#E67E22","#9B59B6","#2ECC71","#E74C3C"]
for n, cl in zip(names, clrs):
    axes[1].scatter(latency_map[n], s1_test[n]["f1_macro"],
                    s=200, color=cl, zorder=5, edgecolors="white", linewidth=1.5, label=n)
    axes[1].annotate(n, (latency_map[n], s1_test[n]["f1_macro"]),
                     textcoords="offset points", xytext=(6,4), fontsize=10, fontweight="bold")
axes[1].set_xscale("log"); axes[1].set_xlabel("Latency (μs, log scale)")
axes[1].set_ylabel("F1-Macro"); axes[1].legend(fontsize=9)
axes[1].set_title("Latency vs Accuracy", fontweight="bold")
plt.suptitle("Stage 1 — Binary Classification", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT/"stage1_comparison.png", dpi=120, bbox_inches="tight")
plt.show(); print("Saved stage1_comparison.png")


✅ Saved stage1_predictions.csv  (60,000 rows × 28 cols)
Saved stage1_comparison.png


In [ ]:
# ── Confusion matrices ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for ax, name in zip(axes, names):
    cm = confusion_matrix(y_test, s1_test[name]["y_pred"])
    ConfusionMatrixDisplay(cm, display_labels=["Attack","Normal"]).plot(
        ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(name, fontweight="bold")
plt.suptitle("Stage 1 Confusion Matrices — Test Set", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT/"stage1_confusion_matrices.png", dpi=120, bbox_inches="tight")
plt.show(); print("Saved stage1_confusion_matrices.png")


## Section 5 — Stage 2: Attack Type Identification
Only samples **predicted as Attack (0)** by Stage 1 proceed to Stage 2.  
Stage 2 classifies which specific attack type it is.


In [ ]:
if y2_sub is None:
    print("⚠️  Attack_subType not found in df2 — skipping Stage 2")
else:
    # ── HGB Stage 2 ───────────────────────────────────────────────────────
    print("Training Stage 2 HGB (attack subtype classifier) …")
    t0 = time.time()
    hgb_s2 = HistGradientBoostingClassifier(
        max_iter=200, max_depth=10, learning_rate=0.05, random_state=RANDOM_STATE)
    hgb_s2.fit(X2tr_s, y2tr_enc)
    hgb_s2_time = time.time() - t0

    y2te_pred_hgb = hgb_s2.predict(X2te_s)
    hgb_s2_f1     = f1_score(y2te_enc, y2te_pred_hgb, average="macro")
    print(f"HGB Stage-2 → F1-Macro: {hgb_s2_f1:.4f}  Train: {hgb_s2_time:.1f}s")
    print("\nStage 2 HGB Report:")
    print(classification_report(y2te_enc, y2te_pred_hgb, target_names=le.classes_))


In [ ]:
if y2_sub is not None:
    # ── CNN Stage 2 ───────────────────────────────────────────────────────
    cnn_s2 = build_cnn_multiclass(X2tr_s.shape[1], N_CLASSES2)
    cnn_s2.summary()

    X2tr_cnn  = X2tr_s.reshape(-1, X2tr_s.shape[1], 1)
    X2val_cnn = X2val_s.reshape(-1, X2val_s.shape[1], 1)
    X2te_cnn  = X2te_s.reshape(-1,  X2te_s.shape[1], 1)

    # Class weights for multiclass
    cw2 = compute_class_weight("balanced", classes=np.arange(N_CLASSES2), y=y2tr_enc)
    cwd2 = {i: float(w) for i, w in enumerate(cw2)}

    cnn_s2_callbacks = [
        K_callbacks.EarlyStopping(patience=4, restore_best_weights=True,
                                  monitor="val_accuracy", mode="max"),
        K_callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                      patience=2, min_lr=1e-5),
    ]

    print("Training Stage 2 CNN …")
    t0 = time.time()
    hist_s2 = cnn_s2.fit(
        X2tr_cnn, y2tr_enc,
        validation_data=(X2val_cnn, y2val_enc),
        epochs=30, batch_size=1024,
        class_weight=cwd2, callbacks=cnn_s2_callbacks, verbose=1,
    )
    cnn_s2_time = time.time() - t0
    print(f"\nCNN Stage-2 training: {cnn_s2_time:.1f}s  ({len(hist_s2.epoch)} epochs)")

    cnn_s2_pred  = np.argmax(cnn_s2.predict(X2te_cnn, verbose=0), axis=1)
    cnn_s2_f1    = f1_score(y2te_enc, cnn_s2_pred, average="macro")
    print(f"CNN Stage-2 → F1-Macro: {cnn_s2_f1:.4f}")
    print("\nStage 2 CNN Report:")
    print(classification_report(y2te_enc, cnn_s2_pred, target_names=le.classes_))


In [ ]:
if y2_sub is not None:
    # ── Stage 2 Training Curves ───────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(hist_s2.history["loss"],         label="train", color="#E74C3C")
    axes[0].plot(hist_s2.history["val_loss"],      label="val",   color="#3498DB")
    axes[0].set_title("CNN Stage 2 — Loss",        fontweight="bold"); axes[0].legend()
    axes[1].plot(hist_s2.history["accuracy"],      label="train", color="#E74C3C")
    axes[1].plot(hist_s2.history["val_accuracy"],  label="val",   color="#3498DB")
    axes[1].set_title("CNN Stage 2 — Accuracy",    fontweight="bold"); axes[1].legend()
    plt.suptitle("Stage 2 CNN Training Curves", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(OUT/"cnn_stage2_training_curves.png", dpi=120, bbox_inches="tight")
    plt.show(); print("Saved cnn_stage2_training_curves.png")

    # ── Stage 2 Confusion Matrix (CNN) ───────────────────────────────────
    cm_s2 = confusion_matrix(y2te_enc, cnn_s2_pred)
    fig, ax = plt.subplots(figsize=(11, 8))
    sns.heatmap(cm_s2, annot=True, fmt="d", cmap="Blues",
                xticklabels=le.classes_, yticklabels=le.classes_, ax=ax)
    ax.set_title("Stage 2 CNN — Confusion Matrix (Attack Subtypes)", fontweight="bold")
    plt.xticks(rotation=45, ha="right"); plt.tight_layout()
    plt.savefig(OUT/"stage2_cnn_confusion_matrix.png", dpi=120, bbox_inches="tight")
    plt.show(); print("Saved stage2_cnn_confusion_matrix.png")


In [ ]:
if y2_sub is not None:
    # ════════════════════════════════════════════════════════════
    # SAVE: stage2_subtype_report.csv
    # ════════════════════════════════════════════════════════════
    rpt_cnn  = classification_report(y2te_enc, cnn_s2_pred,
                   target_names=le.classes_, output_dict=True)
    rpt_hgb  = classification_report(y2te_enc, y2te_pred_hgb,
                   target_names=le.classes_, output_dict=True)

    sub_rows = []
    for cls, mets in rpt_cnn.items():
        if isinstance(mets, dict):
            hgb_m = rpt_hgb.get(cls, {})
            sub_rows.append({
                "class":          cls,
                "cnn_precision":  round(mets["precision"], 4),
                "cnn_recall":     round(mets["recall"],    4),
                "cnn_f1":         round(mets["f1-score"],  4),
                "hgb_precision":  round(hgb_m.get("precision",0), 4),
                "hgb_recall":     round(hgb_m.get("recall",0),    4),
                "hgb_f1":         round(hgb_m.get("f1-score",0),  4),
                "support":        mets["support"],
            })
    df_sub = pd.DataFrame(sub_rows)
    df_sub.to_csv(OUT/"stage2_subtype_report.csv", index=False)
    print("✅ Saved stage2_subtype_report.csv")
    print(df_sub.to_string(index=False))


In [ ]:
if y2_sub is not None:
    # ════════════════════════════════════════════════════════════
    # SAVE: stage2_predictions.csv  (CNN predictions)
    # ════════════════════════════════════════════════════════════
    s2_pred_df = X2te.copy().reset_index(drop=True)
    s2_pred_df["true_subtype_encoded"]  = y2te_enc
    s2_pred_df["true_subtype"]          = le.inverse_transform(y2te_enc)
    s2_pred_df["cnn_pred_encoded"]      = cnn_s2_pred
    s2_pred_df["cnn_pred_subtype"]      = le.inverse_transform(cnn_s2_pred)
    s2_pred_df["hgb_pred_encoded"]      = y2te_pred_hgb
    s2_pred_df["hgb_pred_subtype"]      = le.inverse_transform(y2te_pred_hgb)
    s2_pred_df["cnn_correct"]           = (cnn_s2_pred == y2te_enc).astype(int)
    s2_pred_df["hgb_correct"]           = (y2te_pred_hgb == y2te_enc).astype(int)
    s2_pred_df.to_csv(OUT/"stage2_predictions.csv", index=False)
    print(f"✅ Saved stage2_predictions.csv  ({len(s2_pred_df):,} rows)")


## Section 6 — Full Two-Stage Pipeline (End-to-End)
Run every test sample through Stage 1 → if attack, run through Stage 2.


In [ ]:
if y2_sub is not None:
    # Use best Stage-1 model (RF or HGB) + CNN Stage 2
    # Pick whichever Stage-1 model has the best F1-Macro
    best_s1_name = max(s1_test, key=lambda n: s1_test[n]["f1_macro"])
    print(f"Best Stage-1 model: {best_s1_name}  "
          f"(F1-Macro={s1_test[best_s1_name]['f1_macro']:.4f})")

    # Stage 1 predictions on test set (using best model)
    s1_best_pred  = s1_test[best_s1_name]["y_pred"]
    s1_best_proba = s1_test[best_s1_name]["y_proba"]

    # Stage 2: only run CNN on samples Stage-1 flagged as attack
    attack_mask = (s1_best_pred == 0)
    print(f"Samples flagged as ATTACK by Stage 1: {attack_mask.sum():,} "
          f"({attack_mask.mean()*100:.1f}%)")

    # Prepare Stage-2 input for attacked samples
    X_test_attack = X_test_s[attack_mask]
    X_test_attack_cnn = X_test_attack.reshape(-1, X_test_attack.shape[1], 1)

    # Predict attack subtype via CNN
    if len(X_test_attack_cnn) > 0:
        s2_probs = cnn_s2.predict(X_test_attack_cnn, verbose=0)
        s2_preds = np.argmax(s2_probs, axis=1)
        s2_labels = le.inverse_transform(s2_preds)
    else:
        s2_labels = np.array([])

    # ── Assemble full pipeline result ────────────────────────────
    pipeline_df = X_test.copy().reset_index(drop=True)
    pipeline_df["true_binary_label"]  = y_test.values
    pipeline_df["stage1_pred"]        = s1_best_pred
    pipeline_df["stage1_proba"]       = s1_best_proba.round(4)
    pipeline_df["stage1_decision"]    = np.where(s1_best_pred == 0, "ATTACK", "NORMAL")

    # Fill Stage-2 results
    pipeline_df["stage2_attack_type"] = "N/A"
    attack_indices = np.where(attack_mask)[0]
    for i, label in zip(attack_indices, s2_labels):
        pipeline_df.at[i, "stage2_attack_type"] = label

    # Final combined label
    pipeline_df["final_prediction"] = pipeline_df.apply(
        lambda r: r["stage2_attack_type"]
                  if r["stage1_decision"] == "ATTACK"
                  else "NORMAL", axis=1)

    # ════════════════════════════════════════════════════════════
    # SAVE: pipeline_combined_predictions.csv
    # ════════════════════════════════════════════════════════════
    pipeline_df.to_csv(OUT/"pipeline_combined_predictions.csv", index=False)
    print(f"\n✅ Saved pipeline_combined_predictions.csv  ({len(pipeline_df):,} rows)")

    print("\nFinal prediction distribution:")
    print(pipeline_df["final_prediction"].value_counts().to_string())


In [ ]:
if y2_sub is not None:
    # ── Pipeline accuracy breakdown ───────────────────────────────────────
    # For samples where Stage 2 ran, compare to true subtype
    # (Note: true subtype is only available in df2, not df1 test split,
    #  so we show Stage-1 accuracy on the binary test set as the main metric)

    print("\n── PIPELINE SUMMARY ─────────────────────────────────────")
    print(f"  Stage 1 ({best_s1_name}) F1-Macro  : {s1_test[best_s1_name]['f1_macro']:.4f}")
    print(f"  Stage 1 ({best_s1_name}) AUC-ROC   : {s1_test[best_s1_name]['auc_roc']:.4f}")
    print(f"  Stage 2 CNN F1-Macro     : {cnn_s2_f1:.4f}")
    print(f"  Stage 2 HGB F1-Macro     : {hgb_s2_f1:.4f}")
    print(f"  Samples reaching Stage 2 : {attack_mask.sum():,} / {len(s1_best_pred):,}")
    print(f"  Normal (stopped at S1)   : {(~attack_mask).sum():,} / {len(s1_best_pred):,}")

    # ── Sankey-style flow diagram ─────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.set_xlim(0, 10); ax.set_ylim(0, 6); ax.axis("off")
    ax.set_title("Two-Stage Pipeline Flow (test set)", fontsize=14, fontweight="bold")

    total = len(s1_best_pred)
    n_atk = int(attack_mask.sum())
    n_nrm = total - n_atk

    # Boxes
    for (bx, by, bw, bh, col, lbl) in [
        (0.3, 2.0, 2.5, 2.0, "#1E3A5F", f"Test Set\n{total:,} samples"),
        (3.5, 3.2, 2.5, 1.5, "#C0392B", f"Attack\n{n_atk:,} samples"),
        (3.5, 0.8, 2.5, 1.5, "#1A7A4A", f"Normal\n{n_nrm:,} samples"),
        (6.8, 3.2, 2.8, 1.5, "#8E44AD", f"Stage 2 CNN\n{N_CLASSES2} subtypes"),
    ]:
        ax.add_patch(plt.Rectangle((bx, by), bw, bh, color=col, alpha=0.85, zorder=2))
        ax.text(bx+bw/2, by+bh/2, lbl, ha="center", va="center",
                color="white", fontsize=10, fontweight="bold", zorder=3)

    # Arrows
    ax.annotate("", xy=(3.5,4.0), xytext=(2.8,3.5),
                arrowprops=dict(arrowstyle="->", color="#C0392B", lw=2))
    ax.annotate("", xy=(3.5,1.5), xytext=(2.8,2.8),
                arrowprops=dict(arrowstyle="->", color="#1A7A4A", lw=2))
    ax.annotate("", xy=(6.8,4.0), xytext=(6.0,4.0),
                arrowprops=dict(arrowstyle="->", color="#8E44AD", lw=2))

    ax.text(1.55, 5.3, "Stage 1\nBinary Classifier", ha="center",
            color="#00D4FF", fontsize=9, fontweight="bold")
    ax.add_patch(plt.Rectangle((0.3, 4.8), 2.5, 0.9, color="#0D2137", alpha=0.7, zorder=2))
    ax.text(1.55, 5.25, f"Best: {best_s1_name}  F1={s1_test[best_s1_name]['f1_macro']:.4f}",
            ha="center", color="white", fontsize=8, zorder=3)

    plt.tight_layout()
    plt.savefig(OUT/"pipeline_flow_diagram.png", dpi=120, bbox_inches="tight")
    plt.show(); print("Saved pipeline_flow_diagram.png")


## Section 7 — Feature Importance

In [ ]:
rf_model = s1_val["RF"]["model"]
imp_rf   = pd.Series(rf_model.feature_importances_, index=USE_FEATS).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 7))
imp_rf.sort_values().plot(kind="barh", ax=ax,
    color=["#E74C3C" if v > imp_rf.median() else "#85C1E9" for v in imp_rf.sort_values()],
    edgecolor="white")
ax.set_title("Random Forest Feature Importances", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT/"feature_importance_rf.png", dpi=120, bbox_inches="tight")
plt.show()

# ════════════════════════════════════════════════════════════
# SAVE: feature_importance_rf.csv
# ════════════════════════════════════════════════════════════
fi_df = imp_rf.reset_index()
fi_df.columns = ["feature", "importance"]
fi_df["rank"] = range(1, len(fi_df)+1)
fi_df.to_csv(OUT/"feature_importance_rf.csv", index=False)
print("✅ Saved feature_importance_rf.csv")
print(fi_df.head(10).to_string(index=False))


## Section 8 — Per-Device Evaluation

In [ ]:
if "Device_Name" in df2.columns and "Attack" in df2.columns:
    device_rows = []
    use2_cols = [f for f in USE_FEATS if f in df2.columns]

    for device in sorted(df2["Device_Name"].unique()):
        mask  = df2["Device_Name"] == device
        X_dev = scaler1.transform(log1p_tf(df2.loc[mask, use2_cols], SKEW_FEATS))
        y_dev = df2.loc[mask, "label"]
        if len(y_dev.unique()) < 2: continue

        # Use best Stage-1 sklearn model for device eval
        best_sk = max((n for n in s1_val if n != "CNN"),
                      key=lambda n: s1_val[n]["f1_macro"])
        ydev_pred = s1_val[best_sk]["model"].predict(X_dev)
        device_rows.append({
            "device":    device,
            "model":     best_sk,
            "f1_attack": round(f1_score(y_dev, ydev_pred, pos_label=0, zero_division=0), 4),
            "f1_normal": round(f1_score(y_dev, ydev_pred, pos_label=1, zero_division=0), 4),
            "n_samples": int(mask.sum()),
        })

    df_dev = pd.DataFrame(device_rows).sort_values("f1_attack", ascending=False)
    df_dev.to_csv(OUT/"per_device_results.csv", index=False)
    print("✅ Saved per_device_results.csv")
    print(df_dev.to_string(index=False))

    fig, ax = plt.subplots(figsize=(12, 5))
    xd = np.arange(len(df_dev))
    ax.bar(xd-0.2, df_dev["f1_attack"], 0.4, label="F1-Attack", color="#E74C3C", alpha=0.85)
    ax.bar(xd+0.2, df_dev["f1_normal"], 0.4, label="F1-Normal", color="#2ECC71", alpha=0.85)
    ax.set_xticks(xd)
    ax.set_xticklabels(df_dev["device"], rotation=30, ha="right", fontsize=8)
    ax.set_ylim(0, 1.05); ax.set_ylabel("F1-Score")
    ax.set_title("Per-Device F1 (Stage 1 Best Model)", fontweight="bold")
    ax.legend(); plt.tight_layout()
    plt.savefig(OUT/"per_device_results.png", dpi=120, bbox_inches="tight")
    plt.show(); print("Saved per_device_results.png")
else:
    print("⚠️  Device_Name not in df2 — skipping")


In [ ]:
# ════════════════════════════════════════════════════════════
# SAVE: cnn_training_history.csv
# ════════════════════════════════════════════════════════════
hist_rows = []
for epoch_i, (loss, vloss, auc, vauc) in enumerate(zip(
        hist_s1.history["loss"], hist_s1.history["val_loss"],
        hist_s1.history["auc"],  hist_s1.history["val_auc"])):
    hist_rows.append({
        "stage": "Stage1_Binary", "epoch": epoch_i+1,
        "train_loss": round(loss, 5), "val_loss": round(vloss, 5),
        "train_auc":  round(auc,  5), "val_auc":  round(vauc,  5),
    })

if y2_sub is not None:
    for epoch_i, (loss, vloss, acc, vacc) in enumerate(zip(
            hist_s2.history["loss"], hist_s2.history["val_loss"],
            hist_s2.history["accuracy"], hist_s2.history["val_accuracy"])):
        hist_rows.append({
            "stage": "Stage2_Multiclass", "epoch": epoch_i+1,
            "train_loss": round(loss, 5), "val_loss": round(vloss, 5),
            "train_auc":  round(acc,  5), "val_auc":  round(vacc,  5),
        })

pd.DataFrame(hist_rows).to_csv(OUT/"cnn_training_history.csv", index=False)
print("✅ Saved cnn_training_history.csv")


## Section 10 — Final Summary

In [ ]:
print("=" * 65)
print("FINAL RESULTS SUMMARY")
print("=" * 65)
print("\n── STAGE 1 (Binary) ──────────────────────────────────────────")
s1_final = pd.DataFrame({
    name: {
        "F1-Attack(0)": round(s1_test[name]["f1_attack"], 4),
        "F1-Normal(1)": round(s1_test[name]["f1_normal"], 4),
        "F1-Macro":     round(s1_test[name]["f1_macro"],  4),
        "AUC-ROC":      round(s1_test[name]["auc_roc"],   4),
        "Latency(μs)":  round(latency_map[name],          3),
    }
    for name in s1_test
}).T
print(s1_final.to_string())

if y2_sub is not None:
    print("\n── STAGE 2 (Attack Subtype) ──────────────────────────────────")
    print(f"  CNN F1-Macro  : {cnn_s2_f1:.4f}")
    print(f"  HGB F1-Macro  : {hgb_s2_f1:.4f}")
    print(f"  Classes       : {list(le.classes_)}")

print("\n── OUTPUT FILES ──────────────────────────────────────────────")
for f in sorted(OUT.glob("*")):
    print(f"  {f.name:50s}  {f.stat().st_size/1e3:6.1f} KB")
